# Detailed Solutions Notebook — Regularization for Wine Quality Classification

This is the fully worked answer key to `02_Skeleton_Practice_Notebook.ipynb`. Every checkpoint is solved with code **and** an explanation of the reasoning — not just "what" but "why." Use it to check your own attempt or to study the workflow end to end.

## Part 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('wine_quality.csv')
print(df.shape)
df.head()

**Why:** `seaborn` is imported alongside `matplotlib` because it gives cleaner default statistical plots (count plots, heatmaps) with less code.

In [ ]:
print(df.isna().sum().sum(), 'missing values total')
sns.countplot(x='quality', data=df)
plt.title('Class balance')
plt.show()
df['quality'].value_counts(normalize=True)

**Answer:** There are no missing values. The classes are ~53% (1) / ~47% (0) — close enough to balanced that accuracy is a reasonable headline metric, but F1 remains the more robust choice since it accounts for both false positives and false negatives and is what the original lesson script reports.

## Part 1 — EDA

In [ ]:
corr = df.corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.tight_layout()
plt.show()

pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
             .stack().rename('corr').reset_index())
pairs[pairs['corr'].abs() > 0.6].sort_values('corr', key=abs, ascending=False)

**Answer:** The strongest relationships are `fixed acidity` with `citric acid` (positive) and `fixed acidity` with `pH` (negative, since more acid lowers pH), and `free sulfur dioxide` with `total sulfur dioxide` (free SO₂ is a subset of total SO₂ by definition, so they move together). These are exactly the kind of collinear feature pairs that make an unregularized model's coefficients unstable — small changes in the training sample can swing which of the two correlated features "gets credit," which is one textbook symptom of overfitting.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.ravel(), df.columns):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Part 2 — Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler

y = df['quality']
features = df.drop(columns=['quality'])
predictors = features.columns

X = StandardScaler().fit_transform(features)

**Why scaling matters here:** the L1 penalty is `α·Σ|bⱼ|` and the L2 penalty is `α·Σbⱼ²` — both are sums over raw coefficient values. `total sulfur dioxide` ranges into the hundreds while `chlorides` sits around 0.08; to explain the same amount of variance in the target, the `chlorides` coefficient would naturally be numerically much larger than the `total sulfur dioxide` coefficient just to compensate for the units. An unscaled penalty would then punish `chlorides` far more harshly than `total sulfur dioxide` for no principled reason. `StandardScaler` puts every feature on a mean-0, std-1 footing so the penalty compares coefficients on equal footing — reflecting genuine relative importance rather than an artifact of measurement units.

## Part 3 — Train/test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=99, stratify=y
)
print(X_train.shape, X_test.shape)

**Why `stratify=y`:** with a target that's already fairly balanced this matters less, but it's a habit worth building — it guarantees the train and test sets both keep roughly the same 53/47 split, so a lucky or unlucky split can't distort the evaluation.

## Part 4 — Baseline: no regularization

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

clf_no_reg = LogisticRegression(penalty=None, max_iter=5000)
clf_no_reg.fit(X_train, y_train)

print('Training F1:', f1_score(y_train, clf_no_reg.predict(X_train)))
print('Testing  F1:', f1_score(y_test, clf_no_reg.predict(X_test)))

In [ ]:
coef = pd.Series(clf_no_reg.coef_.ravel(), predictors).sort_values()
coef.plot(kind='bar', title='Coefficients (no regularization)')
plt.axhline(0, color='k', linewidth=0.8)
plt.tight_layout()
plt.show()

**Observations:** the training F1 is noticeably higher than the test F1 — the classic overfitting signature. `volatile acidity` and `alcohol` tend to dominate the coefficient plot, while the collinear pairs identified in Part 1 (e.g., the two sulfur-dioxide features) show up with coefficients that partly cancel each other out — a sign the model is splitting credit between correlated features somewhat arbitrarily, which is exactly the instability regularization is meant to fix.

## Part 5 — Default logistic regression (L2)

In [ ]:
clf_default = LogisticRegression(max_iter=5000)  # penalty='l2', C=1.0 by default
clf_default.fit(X_train, y_train)

print('Ridge training F1:', f1_score(y_train, clf_default.predict(X_train)))
print('Ridge testing  F1:', f1_score(y_test, clf_default.predict(X_test)))

**What changed:** training F1 typically drops slightly relative to Part 4 (we've added bias on purpose), while test F1 holds steady or improves slightly — the model is trading a little bit of training fit for better generalization, which is the entire point of regularization.

## Part 6 — Coarse hyperparameter search

In [ ]:
C_array = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]
training_array, test_array = [], []
for c in C_array:
    clf = LogisticRegression(C=c, max_iter=5000)
    clf.fit(X_train, y_train)
    training_array.append(f1_score(y_train, clf.predict(X_train)))
    test_array.append(f1_score(y_test, clf.predict(X_test)))

plt.plot(C_array, training_array, marker='o', label='Training F1')
plt.plot(C_array, test_array, marker='o', label='Test F1')
plt.xscale('log')
plt.xlabel('C')
plt.legend()
plt.show()

**Reading the plot:** at very small `C` (strong regularization) both curves sit lower — the model is underfitting. As `C` increases, both curves rise and test performance plateaus somewhere around `C≈1–10` before the training curve keeps climbing while test performance flattens or dips slightly — this plateau is the region worth searching finely in Part 7.

## Part 7 — Fine-grained search with GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

C_array_fine = np.logspace(-3, 2, 100)
gs = GridSearchCV(LogisticRegression(max_iter=5000), param_grid={'C': C_array_fine}, scoring='f1', cv=5)
gs.fit(X_train, y_train)
print(gs.best_params_, gs.best_score_)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

clf_best = LogisticRegression(C=gs.best_params_['C'], max_iter=5000)
clf_best.fit(X_train, y_train)
print('Test F1 with tuned C:', f1_score(y_test, clf_best.predict(X_test)))

ConfusionMatrixDisplay.from_estimator(clf_best, X_test, y_test)
plt.show()

**Why 5-fold CV instead of a single train/test split for tuning:** a single split's `best C` could be an artifact of which rows happened to land in the test set. Cross-validation averages performance over 5 different splits, so the chosen `C` is more likely to generalize rather than to have overfit to one particular test set.

## Part 8 — L1 (Lasso) with LogisticRegressionCV

In [ ]:
from sklearn.linear_model import LogisticRegressionCV

clf_l1 = LogisticRegressionCV(
    Cs=np.logspace(-2, 2, 100), cv=5, penalty='l1', solver='liblinear',
    scoring='f1', max_iter=5000
)
clf_l1.fit(X, y)
print('Best C:', clf_l1.C_)

In [ ]:
coef_l1 = pd.Series(clf_l1.coef_.ravel(), predictors).sort_values()
coef_l1.plot(kind='bar', title='Tuned L1 coefficients')
plt.axhline(0, color='k', linewidth=0.8)
plt.tight_layout()
plt.show()

zeroed = coef_l1[coef_l1 == 0]
print(f'{len(zeroed)} of {len(coef_l1)} features zeroed:', list(zeroed.index))

**Comparison to Part 4:** L1 typically zeroes out one member of each collinear pair identified in Part 1 (for example, one of the two sulfur-dioxide features) while keeping the other — this is exactly the mechanism described in the lesson: L1's diamond-shaped constraint region has corners on the axes, so the optimum often lands with one coordinate at exactly zero. This is a *selection*, not proof the zeroed feature is meaningless — with a different random seed or a slightly different sample, L1 could just as easily have kept the other member of the pair instead.

## Part 9 — Extension: full model comparison

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

models = {'No reg': clf_no_reg, 'L2 default': clf_default, 'L2 tuned': clf_best}
rows = []
for name, m in models.items():
    pred = m.predict(X_test)
    proba = m.predict_proba(X_test)[:, 1]
    rows.append({'Model': name, 'Accuracy': accuracy_score(y_test, pred),
                 'Precision': precision_score(y_test, pred), 'Recall': recall_score(y_test, pred),
                 'F1': f1_score(y_test, pred), 'ROC-AUC': roc_auc_score(y_test, proba)})

pred_l1 = clf_l1.predict(X_test)
rows.append({'Model': 'L1 tuned (fit on full data — optimistic)',
             'Accuracy': accuracy_score(y_test, pred_l1),
             'Precision': precision_score(y_test, pred_l1), 'Recall': recall_score(y_test, pred_l1),
             'F1': f1_score(y_test, pred_l1),
             'ROC-AUC': roc_auc_score(y_test, clf_l1.predict_proba(X_test)[:, 1])})

pd.DataFrame(rows).set_index('Model').round(3)

**Findings:** the tuned L2 model usually edges out the untuned default on test F1, and both regularized models beat the unregularized baseline on the test set despite scoring lower on training data — direct evidence that the bias we introduced was worth it. The L1 row is flagged as optimistic because it was fit on the full `(X, y)`, so its test rows were part of its own training data; treat it as an upper bound, not a fair comparison.

## Part 10 — Reflection (sample answers)

1. **Which model to deploy:** the tuned L2 model is a defensible default — it edges out the untrained baseline on F1/ROC-AUC without sacrificing any features, and unlike L1 its feature ranking is stable across resampling. If the downstream use case specifically benefits from a sparse, easy-to-explain model (e.g., "only these 5 measurements matter"), L1 becomes preferable despite the slightly less stable feature selection.

2. **`alpha` vs. `C`:** they are reciprocals in effect but opposite in direction — turning `alpha` *up* strengthens regularization (used with `Ridge`/`Lasso`), while turning `C` *up* weakens it (used with `LogisticRegression`). This inversion exists because `LogisticRegression` parameterizes regularization strength as "how much freedom" the coefficients get, rather than "how much penalty" is applied.

3. **Multi-class target:** `LogisticRegression` handles more than two classes natively (it fits one-vs-rest or multinomial regression depending on solver), but `.coef_` becomes a `(n_classes, n_features)` array instead of a flat vector, so the coefficient bar-plot code needs a small adjustment (e.g., plot one class's row at a time, or use a grouped bar chart). Metrics like `f1_score` also need an `average=` parameter (e.g., `'macro'` or `'weighted'`) since there's no single positive class anymore.